# Category Tree - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.retailrocket_category_tree"
target_table = f"{catalog}.silver.retailrocket_category_tree"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- categoryid: string (nullable = true)
 |-- parentid: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

categoryid,parentid,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1016,213,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
809,169,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
570,9,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1691,885,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
536,1691,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
231,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
542,378,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1146,542,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1140,542,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1479,1537,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 1669
Number of columns: 9


In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


In [0]:
for column, dtype in bronze_df.dtypes[:2]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

categoryid
Null count: 0
Distinct count: 1669
Extra whitespace row count: 0
--------------------
parentid
Null count: 25
Distinct count: 362
Extra whitespace row count: 0
--------------------


categoryid is complete and unique, so it is the key.

In [0]:
distinct_categoryid = bronze_df.select("categoryid").distinct()
distinct_parentid = bronze_df.filter(col("parentid").isNotNull()).select("parentid").distinct()

print(distinct_parentid.subtract(distinct_categoryid).count())

0


In [0]:
print(bronze_df.filter(col("categoryid") == col("parentid")).count())

0


- Every non-null parentid matches an existing categoryid.
- No category points to itself.

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed("categoryid", "category_id")
    .withColumnRenamed("parentid", "parent_category_id")
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- category_id: string (nullable = true)
 |-- parent_category_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

category_id,parent_category_id,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1016,213,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
809,169,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
570,9,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1691,885,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
536,1691,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
231,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
542,378,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1146,542,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1140,542,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1479,1537,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 1669
Silver row count: 1669
